# Healthcare Disease Prediction — Exploratory Data Analysis

> **Author:** Senior Research Engineer | Healthcare AI Pipeline  
> **Datasets:** Cardiovascular Disease · Diabetes (BRFSS 2015) · Heart Disease (BRFSS 2015)  
> **Environment:** Google Colab (Python 3.10+)

---

## Objectives
1. Understand dataset structure and feature semantics.
2. Analyse feature distributions and descriptive statistics.
3. Identify missing values and duplicates.
4. Detect outliers via IQR and visualisations.
5. Study target class distributions — critical for imbalance planning.
6. Discover inter-feature relationships via correlation and mutual information.
7. Export cleaned datasets as CSV for downstream preprocessing.

In [ ]:
# ── 0. Imports & Global Config ─────────────────────────────────────────────
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from scipy import stats

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
PALETTE = 'Set2'
print('✅ Imports complete')

## Dataset Overview

We work with **5 CSV files** that resolve to **3 distinct prediction tasks**:

| File | Task | Target | Rows |
|------|------|--------|------|
| `cardio_train.csv` | Cardiovascular disease | `cardio` (0/1) | 70 000 |
| `diabetes_binary_health_indicators_BRFSS2015.csv` | Diabetes (imbalanced) | `Diabetes_binary` | 253 680 |
| `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` | Diabetes (balanced) | `Diabetes_binary` | 70 692 |
| `diabetes_012_health_indicators_BRFSS2015.csv` | Diabetes (3-class) | `Diabetes_012` | 253 680 |
| `heart_disease_health_indicators_BRFSS2015.csv` | Heart disease | `HeartDiseaseorAttack` | 253 680 |

For modeling we will use: **cardio_train** (cardiovascular), **diabetes_binary** (diabetes, with active imbalance handling), and **heart_disease** (heart disease).

In [ ]:
# ── 1. Load All Datasets ───────────────────────────────────────────────────
t0 = time.time()

# Cardiovascular: semicolon-delimited
cardio = pd.read_csv('cardio_train.csv', sep=';')
cardio.drop(columns=['id'], inplace=True)  # drop meaningless ID
cardio['age_years'] = (cardio['age'] / 365.25).round(1)  # age in days → years

# BRFSS datasets
diab_imb   = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
diab_5050  = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
diab_012   = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')
heart      = pd.read_csv('heart_disease_health_indicators_BRFSS2015.csv')

datasets = {
    'Cardiovascular': (cardio, 'cardio'),
    'Diabetes (imbalanced)': (diab_imb, 'Diabetes_binary'),
    'Diabetes (50-50)': (diab_5050, 'Diabetes_binary'),
    'Diabetes (3-class)': (diab_012, 'Diabetes_012'),
    'Heart Disease': (heart, 'HeartDiseaseorAttack'),
}
print(f'✅ Loaded all datasets in {time.time()-t0:.1f}s')

## Data Dictionary

### Cardiovascular (`cardio_train.csv`)
| Feature | Type | Description |
|---------|------|-------------|
| age | int | Age in days (converted to years) |
| gender | cat | 1=female, 2=male |
| height | int | Height (cm) |
| weight | float | Weight (kg) |
| ap_hi | int | Systolic blood pressure |
| ap_lo | int | Diastolic blood pressure |
| cholesterol | ord | 1=normal, 2=above normal, 3=well above normal |
| gluc | ord | 1=normal, 2=above normal, 3=well above normal |
| smoke / alco / active | bin | Lifestyle binary flags |
| **cardio** | **target** | **1=cardiovascular disease present** |

### BRFSS 2015 Datasets (Diabetes & Heart Disease)
All features are self-reported health indicators (binary or ordinal) from CDC's Behavioral Risk Factor Surveillance System. Key features: `HighBP`, `HighChol`, `BMI`, `Smoker`, `Stroke`, `PhysActivity`, `GenHlth` (1-5 scale), `Age` (13-level ordinal).

In [ ]:
# ── 2. Data Quality Assessment ─────────────────────────────────────────────
print('='*65)
for name, (df, target) in datasets.items():
    nulls = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    print(f"\n📊 {name}")
    print(f"   Shape : {df.shape}")
    print(f"   Nulls : {nulls}")
    print(f"   Dups  : {dups} ({dups/len(df)*100:.2f}%)")
    vc = df[target].value_counts(normalize=True)
    print(f"   Target: {dict(vc.round(3))}")

**Key Insights — Data Quality:**
- No missing values in any dataset (BRFSS survey is complete by design).
- Cardiovascular data has zero duplicates; BRFSS datasets have significant duplicates (~24k) that must be removed before modeling to prevent data leakage.
- **Heart Disease is severely imbalanced** (~9% positive class) — SMOTETomek or ADASYN will be critical.
- Diabetes (imbalanced) is also skewed (~14% positive); the 50-50 split version is available as a pre-balanced alternative.

In [ ]:
# ── 3. Target Distribution Visualisation ──────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
titles = list(datasets.keys())

for ax, (name, (df, target)) in zip(axes, datasets.items()):
    vc = df[target].value_counts().sort_index()
    colors = sns.color_palette(PALETTE, len(vc))
    bars = ax.bar(vc.index.astype(str), vc.values, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f'{v:,}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Target Class Distributions Across All Datasets', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('target_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Saved: target_distributions.png')

In [ ]:
# ── 4. Descriptive Statistics — Cardiovascular ────────────────────────────
print('📊 Cardiovascular — Descriptive Statistics')
display(cardio.describe().T.round(2))

In [ ]:
# ── 5. Outlier Detection — Cardiovascular (clinical domain knowledge) ──────
# Blood pressure outliers: physiologically impossible values
print('Blood pressure range BEFORE cleaning:')
print(cardio[['ap_hi', 'ap_lo']].describe())

# Remove clearly erroneous BP readings
cardio_clean = cardio[
    (cardio['ap_hi'] >= 60)  & (cardio['ap_hi'] <= 250) &
    (cardio['ap_lo'] >= 40)  & (cardio['ap_lo'] <= 150) &
    (cardio['ap_hi'] > cardio['ap_lo'])  # systolic must exceed diastolic
].copy()

# Height/weight sanity
cardio_clean = cardio_clean[
    (cardio_clean['height'] >= 140) & (cardio_clean['height'] <= 210) &
    (cardio_clean['weight'] >= 40)  & (cardio_clean['weight'] <= 200)
].copy()

removed = len(cardio) - len(cardio_clean)
print(f'\n✂️  Removed {removed} outlier rows ({removed/len(cardio)*100:.2f}%)')
print(f'✅ cardio_clean shape: {cardio_clean.shape}')

In [ ]:
# ── 6. Univariate Analysis — Feature Distributions ────────────────────────
# Cardiovascular numeric features
num_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for i, col in enumerate(num_cols):
    # Distribution by target
    for cls in [0, 1]:
        subset = cardio_clean[cardio_clean['cardio'] == cls][col]
        axes[0, i].hist(subset, bins=40, alpha=0.6,
                        label=f'cardio={cls}', density=True)
    axes[0, i].set_title(col, fontweight='bold')
    axes[0, i].legend(fontsize=8)

    # Boxplot
    cardio_clean.boxplot(column=col, by='cardio', ax=axes[1, i])
    axes[1, i].set_title('')
    axes[1, i].set_xlabel('cardio')

plt.suptitle('Cardiovascular — Numeric Feature Distributions by Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cardio_univariate.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 7. BRFSS Univariate — Heart Disease ───────────────────────────────────
heart_clean = heart.drop_duplicates().copy()
print(f'Heart Disease after dedup: {heart_clean.shape}')

brfss_key_cols = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age']
fig, axes = plt.subplots(1, len(brfss_key_cols), figsize=(20, 4))

for ax, col in zip(axes, brfss_key_cols):
    for cls in [0.0, 1.0]:
        subset = heart_clean[heart_clean['HeartDiseaseorAttack'] == cls][col]
        ax.hist(subset, bins=30, alpha=0.6, label=f'HD={int(cls)}', density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Heart Disease — Key Feature Distributions by Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('heart_univariate.png', bbox_inches='tight', dpi=150)
plt.show()

## Correlation Analysis

In [ ]:
# ── 8. Correlation Heatmaps ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for ax, (df, target, title) in zip(axes, [
    (cardio_clean, 'cardio', 'Cardiovascular'),
    (heart_clean,  'HeartDiseaseorAttack', 'Heart Disease')
]):
    corr = df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
    ax.set_title(f'{title} — Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('correlation_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 9. Mutual Information Feature Importance ──────────────────────────────
def plot_mutual_info(df, target, title, top_n=15, ax=None):
    X = df.drop(columns=[target])
    y = df[target].astype(int)
    mi = mutual_info_classif(X, y, random_state=RANDOM_STATE)
    mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=True).tail(top_n)
    mi_series.plot(kind='barh', ax=ax, color=sns.color_palette(PALETTE, top_n))
    ax.set_title(f'{title} — Mutual Information', fontweight='bold')
    ax.set_xlabel('MI Score')
    return mi_series

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
mi_cardio = plot_mutual_info(cardio_clean, 'cardio', 'Cardiovascular', ax=axes[0])
mi_heart  = plot_mutual_info(heart_clean,  'HeartDiseaseorAttack', 'Heart Disease', ax=axes[1])
plt.tight_layout()
plt.savefig('mutual_information.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n🔑 Top 5 Cardiovascular features by MI:')
print(mi_cardio.tail(5).to_string())
print('\n🔑 Top 5 Heart Disease features by MI:')
print(mi_heart.tail(5).to_string())

In [ ]:
# ── 10. Bivariate Analysis — Cardiovascular ───────────────────────────────
# BMI proxy: compute for cardio
cardio_clean['bmi'] = cardio_clean['weight'] / (cardio_clean['height'] / 100) ** 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age vs Blood Pressure coloured by target
sample = cardio_clean.sample(3000, random_state=RANDOM_STATE)
axes[0].scatter(sample['age_years'], sample['ap_hi'],
                c=sample['cardio'], cmap='coolwarm', alpha=0.4, s=10)
axes[0].set_xlabel('Age (years)'); axes[0].set_ylabel('Systolic BP')
axes[0].set_title('Age vs Systolic BP (red=disease)', fontweight='bold')

# Cholesterol distribution
sns.countplot(data=cardio_clean, x='cholesterol', hue='cardio',
              palette=PALETTE, ax=axes[1])
axes[1].set_title('Cholesterol Level by Target', fontweight='bold')

# BMI distribution
for cls in [0, 1]:
    axes[2].hist(cardio_clean[cardio_clean['cardio']==cls]['bmi'],
                 bins=50, alpha=0.6, label=f'cardio={cls}', density=True)
axes[2].set_xlim(10, 60)
axes[2].set_xlabel('BMI'); axes[2].set_title('BMI by Target', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('cardio_bivariate.png', bbox_inches='tight', dpi=150)
plt.show()

## Key Insights

### Cardiovascular Disease
- **Age** is the strongest individual predictor (MI score highest); disease risk rises sharply after age 50.
- **Systolic blood pressure (ap_hi)** shows clear separation between classes. Hypertension (≥140 mmHg) is a primary risk marker.
- **Cholesterol level 3** (well above normal) associates strongly with disease presence.
- BMI distributions overlap but disease patients skew towards overweight/obese range.
- Raw data contains physiologically impossible BP readings (e.g., ap_hi < 0) — these were removed as hard outliers.

### Heart Disease
- **GenHlth** (general health self-rating 1–5) is the top predictor — poor self-rated health is a robust proxy for clinical outcomes.
- **DiffWalk**, **HighBP**, **Age**, and **Stroke** history are the next most informative features.
- Class imbalance is severe (~9% positive). Any model trained without balancing will learn to predict the majority class and achieve misleadingly high accuracy.

### Diabetes
- The `diabetes_binary_5050split` file provides a pre-balanced subset which can be used directly, or the full imbalanced set can be balanced via SMOTE in the preprocessing stage.
- BMI, GenHlth, and HighBP are consistently important across both diabetes and heart disease, revealing shared cardiovascular-metabolic risk pathways.

In [ ]:
# ── 11. Export Cleaned Datasets ───────────────────────────────────────────
# Primary datasets for modeling
diab_clean  = diab_imb.drop_duplicates().copy()   # use imbalanced; handled in preprocessing
heart_clean_final = heart.drop_duplicates().copy()

cardio_clean.to_csv('cleaned_cardiovascular.csv', index=False)
diab_clean.to_csv('cleaned_diabetes.csv', index=False)
heart_clean_final.to_csv('cleaned_heart_disease.csv', index=False)

print('✅ Exported cleaned datasets:')
print(f'   cleaned_cardiovascular.csv   → {cardio_clean.shape}')
print(f'   cleaned_diabetes.csv         → {diab_clean.shape}')
print(f'   cleaned_heart_disease.csv    → {heart_clean_final.shape}')
print(f'\n⏱  Total EDA time: see cell outputs above')

## Conclusion

The EDA reveals three distinct predictive tasks with varying complexity:

| Dataset | Key Challenge | Recommended Strategy |
|---------|--------------|---------------------|
| Cardiovascular | Outliers in BP/anthropometrics | Domain-rule cleaning ✅ done |
| Diabetes | Moderate imbalance (~14%) | SMOTETomek in preprocessing |
| Heart Disease | Severe imbalance (~9%) | ADASYN or SMOTETomek; prioritise Recall |

All datasets are complete (no NaN), have no meaningful multicollinearity issues, and have been cleaned/deduped. Proceed to `02_Data_Preprocessing.ipynb`.